# Data Cleaning & Visualization Project

## Objective
This project demonstrates a complete data preprocessing and exploratory data analysis workflow using **Pandas, NumPy, Matplotlib, and Seaborn**.

The raw dataset contains realistic data-quality issues including missing values, duplicate records, and numerical outliers.

**Workflow:** Raw Data → Quality Assessment → Cleaning → Validation → EDA → Visualization → Insights


## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")


## 2. Load the Raw Dataset

In [ ]:
DATA_PATH = Path("dataset.csv")
df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())


## 3. Initial Data Exploration

In [ ]:
df.info()


In [ ]:
display(df.describe(include="all").T)
print(f"Duplicate rows: {df.duplicated().sum():,}")


## 4. Data Quality Assessment

Before cleaning, we inspect missing values, duplicates, data types, and unusual numerical observations.


In [ ]:
quality_report = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2),
    "Unique Values": df.nunique()
})
display(quality_report)


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

plt.figure(figsize=(9, 5))
sns.barplot(x=missing.values, y=missing.index)
plt.title("Missing Values by Column")
plt.xlabel("Number of Missing Values")
plt.ylabel("Column")
plt.tight_layout()
plt.show()


## 5. Data Cleaning

- Numerical missing values → median
- Categorical missing values → mode
- Exact duplicate records → removed
- Selected numerical outliers → detected with the IQR method

Outlier bounds are based on `Q1 - 1.5 × IQR` and `Q3 + 1.5 × IQR`.


In [ ]:
cleaned_df = df.copy()

# Missing numerical values
for column in cleaned_df.select_dtypes(include=np.number).columns:
    cleaned_df[column] = cleaned_df[column].fillna(cleaned_df[column].median())

# Missing categorical values
for column in cleaned_df.select_dtypes(exclude=np.number).columns:
    mode = cleaned_df[column].mode()
    if not mode.empty:
        cleaned_df[column] = cleaned_df[column].fillna(mode.iloc[0])

# Remove exact duplicates
before = len(cleaned_df)
cleaned_df = cleaned_df.drop_duplicates().reset_index(drop=True)
print(f"Duplicate rows removed: {before - len(cleaned_df)}")


In [ ]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_columns = ["Annual_Income", "Purchase_Amount"]
outlier_summary = []

for column in outlier_columns:
    lower, upper = iqr_bounds(cleaned_df[column])
    mask = (cleaned_df[column] < lower) | (cleaned_df[column] > upper)

    outlier_summary.append({
        "Column": column,
        "Lower Bound": lower,
        "Upper Bound": upper,
        "Outliers": int(mask.sum())
    })

    cleaned_df = cleaned_df.loc[~mask].copy()

cleaned_df = cleaned_df.reset_index(drop=True)
display(pd.DataFrame(outlier_summary))


## 6. Validation After Cleaning

In [ ]:
final_quality_report = pd.DataFrame({
    "Data Type": cleaned_df.dtypes.astype(str),
    "Missing Values": cleaned_df.isna().sum(),
    "Missing %": (cleaned_df.isna().mean() * 100).round(2),
    "Unique Values": cleaned_df.nunique()
})

display(final_quality_report)
print(f"Final duplicate count: {cleaned_df.duplicated().sum()}")
print(f"Final dataset shape: {cleaned_df.shape}")


## 7. Before vs After Cleaning

In [ ]:
comparison = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Missing Values", "Duplicate Rows"],
    "Before Cleaning": [
        df.shape[0], df.shape[1],
        int(df.isna().sum().sum()),
        int(df.duplicated().sum())
    ],
    "After Cleaning": [
        cleaned_df.shape[0], cleaned_df.shape[1],
        int(cleaned_df.isna().sum().sum()),
        int(cleaned_df.duplicated().sum())
    ]
})
display(comparison)


## 8. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=cleaned_df, x="Purchase_Amount", kde=True)
plt.title("Distribution of Purchase Amount")
plt.xlabel("Purchase Amount (₹)")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()


In [ ]:
category_sales = cleaned_df.groupby("Product_Category")["Purchase_Amount"].sum().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=category_sales.values, y=category_sales.index)
plt.title("Total Purchase Amount by Product Category")
plt.xlabel("Total Purchase Amount (₹)")
plt.ylabel("Product Category")
plt.tight_layout()
plt.show()


In [ ]:
city_sales = cleaned_df.groupby("City")["Purchase_Amount"].sum().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=city_sales.values, y=city_sales.index)
plt.title("Total Purchase Amount by City")
plt.xlabel("Total Purchase Amount (₹)")
plt.ylabel("City")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=cleaned_df, x="Product_Category", y="Purchase_Amount")
plt.title("Purchase Amount by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Purchase Amount (₹)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
numeric_columns = ["Age", "Annual_Income", "Purchase_Amount", "Customer_Rating", "Orders"]

plt.figure(figsize=(9, 7))
sns.heatmap(cleaned_df[numeric_columns].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(
    data=cleaned_df,
    x="Annual_Income",
    y="Purchase_Amount",
    hue="Gender",
    alpha=0.7
)
plt.title("Annual Income vs Purchase Amount")
plt.xlabel("Annual Income (₹)")
plt.ylabel("Purchase Amount (₹)")
plt.tight_layout()
plt.show()


## 9. Key Performance Indicators

In [ ]:
total_customers = cleaned_df["Customer_ID"].nunique()
total_revenue = cleaned_df["Purchase_Amount"].sum()
average_purchase = cleaned_df["Purchase_Amount"].mean()
average_rating = cleaned_df["Customer_Rating"].mean()
total_orders = cleaned_df["Orders"].sum()

print("=" * 55)
print("DATA ANALYSIS SUMMARY")
print("=" * 55)
print(f"Unique Customers       : {total_customers:,}")
print(f"Total Purchase Value   : ₹{total_revenue:,.2f}")
print(f"Average Purchase       : ₹{average_purchase:,.2f}")
print(f"Average Customer Rating: {average_rating:.2f}/5")
print(f"Total Orders           : {total_orders:,}")
print("=" * 55)


## 10. Key Insights

1. Data quality was assessed before preprocessing.
2. Numerical missing values were handled using median imputation.
3. Categorical missing values were handled using the mode.
4. Duplicate records were removed to avoid double-counting.
5. IQR-based outlier detection was applied to income and purchase amount.
6. Category and city analysis highlights purchasing patterns.
7. Distribution and box plots reveal spread and unusual observations.
8. Correlation analysis helps identify associations between numerical variables.

> Correlation indicates association and should not be interpreted as proof of causation.


## 11. Export Cleaned Dataset

In [ ]:
OUTPUT_PATH = Path("cleaned_dataset.csv")
cleaned_df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset exported to: {OUTPUT_PATH}")


## Conclusion

This project demonstrates a complete, reproducible data-cleaning and visualization workflow using Python.

**Technologies:** Python • Pandas • NumPy • Matplotlib • Seaborn • Jupyter Notebook
